In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../src'))
if module_path not in sys.path:
    sys.path.append(module_path)

import vsopy.util
import vsopy.phot

OBJ_NAME=['SA111', 'SA41', 'SA38']
# SESSION_TAG='2024/20240827'
#SESSION_TAG='2024/20240828'
SESSION_TAG='2024/20240903'
IMAGE_ROOT = '/srv/public/img'
WORK_ROOT = '/srv/public'



In [ ]:
obj_name = OBJ_NAME[0]
layout = vsopy.util.WorkLayout(WORK_ROOT)

#settings = vsopy.util.Settings(session_layout.settings_file_path)

session = vsopy.util.Session(tag=SESSION_TAG, name=obj_name)
session_layout = layout.get_session(session)
provider = vsopy.phot.BatchDataProvider(session_layout)
provider.images_.rename_column('filter', 'band')
provider.measured_.rename_column('M', 'm')
provider.images_


In [ ]:
from astropy.table import join

def compose_data(layout, tag, obj_name, band):

    session = vsopy.util.Session(tag=tag, name=obj_name)
    session_layout = layout.get_session(session)
    provider = vsopy.phot.BatchDataProvider(session_layout)
    provider.images_.rename_column('filter', 'band')
    provider.measured_.rename_column('M', 'm')

    data = join(join(provider.images_["image_id", "band", "airmass"],
                    provider.measured_["image_id", "auid", "m"],
                    keys=["image_id"]), provider.sequence_, keys=["auid", "band"])
    dataB = data[data["band"] == band[0]]
    dataV = data[data["band"] == band[1]]

    dataCI =join(join(join(dataB,
                            provider.batch_images_["batch_id", "image_id"],
                            keys=["image_id"])["m", "M", "batch_id", "auid"],
                    join(dataV,
                            provider.batch_images_["batch_id", "image_id"],
                            keys=["image_id"])["m", "M", "batch_id", "auid"],
                    keys=["batch_id", "auid"],
                    table_names=["A", "B"]),
                provider.batches_["batch_id", "airmass"],
                keys=["batch_id"])

    dataCI["CI"] = dataCI["M_A"]["mag"] - dataCI["M_B"]["mag"]
    dataCI["ci"] = dataCI["m_A"]["mag"] - dataCI["m_B"]["mag"]
    dataCI["mM"] = dataCI["m_B"]["mag"] - dataCI["M_B"]["mag"]
    return dataCI["CI", "ci", "mM", "airmass", "auid"]
    # datamM = dataCI.copy()
    # dataCI = dataCI[["CI", "ci", "airmass", "auid"]]
    # datamM["mM"] = datamM["m_B"]["mag"] - datamM["M_B"]["mag"]
    # datamM = datamM[["mM", "CI", "airmass", "auid"]]
    # return dataCI, datamM

tables = [ compose_data(layout, SESSION_TAG, obj_name, band=['B', 'V']) for obj_name in OBJ_NAME ]



In [ ]:
from astropy.table import vstack
dataCI = vstack(tables)
#datamM = vstack(tabMM)
dataCI

In [ ]:
import numpy as np
from uncertainties import unumpy

def regress(dataCI):
    X=np.vstack([np.ones(len(dataCI)),
            dataCI["airmass"],
            dataCI["CI"].value,
            dataCI["airmass"] * dataCI["CI"].value,
            dataCI["CI"].value*dataCI["CI"].value]).T
    #print ("X:", X)
    Q = X.T @ X
    QI = np.linalg.inv(Q)
    XX = QI @ X.T
    #print ("QI:", QI)
    sigma = np.diag(QI)
    #print ("sigma:", sigma)

    def calc_coeffs(X, XX, data, sigma):
        a = XX @ data
        #print ("a:", a)
        s2v = np.sum((X @ a - data)**2) / (len(data) - len(a))
        #print ("s2v:", s2v)
        sigmaa=sigma * s2v
        a_u = unumpy.uarray(a, np.sqrt(sigmaa))
        print ("a_u:", a_u)
        k1v = (-a_u[0]*a_u[3] + a_u[1]*a_u[2])/a_u[2]
        k2v = a_u[3]/a_u[2]
        alpha1 = a_u[0]
        alpha2 = a_u[2]
        return k1v, k2v, alpha1, alpha2

    k1v, k2v, alpha1, alpha2 = calc_coeffs(X, XX, dataCI["mM"].value, sigma)
    k1bv, k2bv, beta1, beta2 = calc_coeffs(X, XX, dataCI["ci"].value, sigma)
    return (k1v, k2v, alpha1, alpha2), (k1bv, k2bv, beta1, beta2)

In [ ]:
kk = regress(dataCI)
kk